In [1]:
from util import import_ragas_custom

import_ragas_custom()

Arquivos copiados com sucesso!


In [2]:
import os

import pandas as pd

from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from datasets import Dataset
from ragas.integrations.llama_index import evaluate
from ragas.testset.prompts import translate_prompts
from ragas.run_config import RunConfig
from ragas.metrics.critique import SUPPORTED_ASPECTS

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    Settings,
    load_index_from_storage,
)

from ragas.metrics import (
    answer_relevancy,
    answer_correctness,
    answer_similarity,
    context_precision,
    context_recall,
    context_utilization,
    context_entity_recall,
    noise_sensitivity_irrelevant,
    noise_sensitivity_relevant,
    faithfulness
)


In [3]:
DATA_PATH = 'data'
TESTSET = 'testset_openai.csv'
PERSIST_DIR = "./storage"
LANGUAGE = 'pt'
LANGUAGE_CACHE = 'cache'
TIMEOUT = 2400
RESULT_CSV = 'result_opennai_llama3_2_3b.csv'
MODEL = 'llama3.2'

metrics = [
    answer_relevancy,
    answer_correctness,
    answer_similarity,
    context_precision,
    context_recall,
    context_utilization,
    context_entity_recall,
    noise_sensitivity_irrelevant,
    noise_sensitivity_relevant,
    faithfulness
]

# metrics.extend(SUPPORTED_ASPECTS)


In [4]:
testset = pd.read_csv(TESTSET, index_col=0)
print("Tamanho do dataset: ", len(testset))
print(testset.head())

Tamanho do dataset:  119
                                            question  \
0  Quais são as vantagens de contar com suporte t...   
1  Qual é a função do software IP Power em relaçã...   
2  Qual é a importância da comunicação entre os N...   
3  Qual é a função da corrente contínua no proces...   
4  Qual é a função do Adaptador SNMP no gerenciam...   

                                            contexts  \
0  ['Inovação, qualidade, tecnologia\ne confiabil...   
1  ['ser , monit orar \no status e en viar aler t...   
2  ['S I S T E M A PA R A L E L O M U LT I AT I V...   
3  ['Retificador\nConverte a ener gia da r ede el...   
4  ['REGISTRO PERMANENTE\nDE E VENT OSCORREÇÃO DO...   

                                        ground_truth evolution_type  \
0  As vantagens de contar com suporte técnico 24 ...         simple   
1  O software IP Power é uma ferramenta de gerenc...         simple   
2  A comunicação entre os No Breaks no sistema Pa...         simple   
3  A resposta à p

In [5]:
nan_rows = testset[testset.isna().any(axis=1)]
print("Quantidade de nulos: ", len(nan_rows))
print(nan_rows)

del nan_rows

Quantidade de nulos:  0
Empty DataFrame
Columns: [question, contexts, ground_truth, evolution_type, metadata, episode_done]
Index: []


In [6]:
testset = testset.dropna()

In [7]:
print("Tamanho do dataset: ", len(testset))
print(testset.head())


Tamanho do dataset:  119
                                            question  \
0  Quais são as vantagens de contar com suporte t...   
1  Qual é a função do software IP Power em relaçã...   
2  Qual é a importância da comunicação entre os N...   
3  Qual é a função da corrente contínua no proces...   
4  Qual é a função do Adaptador SNMP no gerenciam...   

                                            contexts  \
0  ['Inovação, qualidade, tecnologia\ne confiabil...   
1  ['ser , monit orar \no status e en viar aler t...   
2  ['S I S T E M A PA R A L E L O M U LT I AT I V...   
3  ['Retificador\nConverte a ener gia da r ede el...   
4  ['REGISTRO PERMANENTE\nDE E VENT OSCORREÇÃO DO...   

                                        ground_truth evolution_type  \
0  As vantagens de contar com suporte técnico 24 ...         simple   
1  O software IP Power é uma ferramenta de gerenc...         simple   
2  A comunicação entre os No Breaks no sistema Pa...         simple   
3  A resposta à p

In [8]:
testset = Dataset.from_pandas(testset)

In [9]:
translate_prompts(LANGUAGE, LANGUAGE_CACHE)

In [10]:
embeding = OllamaEmbedding(model_name=MODEL)
model = Ollama(model=MODEL, request_timeout=TIMEOUT)

Settings.embed_model = embeding
Settings.llm = model

In [11]:
if not os.path.exists(PERSIST_DIR):
    documents = SimpleDirectoryReader(DATA_PATH).load_data()
    index = VectorStoreIndex.from_documents(documents, show_progress=True)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

query_engine = index.as_query_engine(request_timeout=TIMEOUT)

In [12]:
result = evaluate(
    query_engine=query_engine,
    metrics=metrics,
    dataset=testset,
    llm=model,
    embeddings=embeding,
    run_config=RunConfig(timeout=TIMEOUT, max_workers=2)
)

Running Query Engine:   0%|          | 0/119 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1190 [00:00<?, ?it/s]

n values greater than 1 not support for LlamaIndex LLMs
Failed to parse output. Returning None.


-------------Prompt-------------


Dado um texto, extraia entidades únicas sem repetição. Certifique-se de considerar diferentes formas ou menções da mesma entidade como uma única entidade.

A saída deve ser uma instância JSON bem formatada que esteja em conformidade com o esquema JSON abaixo.

Como exemplo, para o esquema {"motive": "string", "note": 0} o objeto {"motive": "Knife", "note": 7}.

Aqui está o esquema JSON de saída:
```
{"entities": ["string"]}
```

Não retorne nenhum preâmbulo ou explicação, retorne apenas uma string JSON pura cercada por acentos graves triplos (```).

Examples:

text: "A Torre Eiffel, localizada em Paris, França, é um dos marcos mais icônicos do mundo.\nMilhões de visitantes são atraídos a ela todos os anos por suas vistas de tirar o fôlego da cidade.\nConcluída em 1889, foi construída a tempo para a Feira Mundial de 1889."
output: ```{"entities": ["Eiffel 

In [13]:
result_dataframe = result.to_pandas()
result_dataframe.to_csv(RESULT_CSV)

In [14]:
print(result)

{'answer_relevancy': 0.7222, 'answer_correctness': 0.5423, 'answer_similarity': 0.8205, 'context_precision': 0.8131, 'context_recall': 0.6702, 'context_utilization': 0.7228, 'context_entity_recall': 0.0550, 'noise_sensitivity_irrelevant': 0.0000, 'noise_sensitivity_relevant': 0.1786, 'faithfulness': 0.7133}
